In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

#read tables

sales_orders_df = spark.read.table("ecommerce_analytics.silver.sales_orders")
order_products_df = spark.read.table("ecommerce_analytics.silver.order_products")
promotions_df = spark.read.table("ecommerce_analytics.silver.promotions")

##STEP 1: Understand Granularity

*  fact_sales should contain:
*  One row for each product inside each order
#
##### Example:
* Order 101 has 3 products
*  --> then fact table = 3 rows
#
*  Therefore BASE table should be:
*  order_products_df
#
##### Why?
* Because it already has:
* order_number + product + qty


### Step 2 : Clean col first

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

order_products_clean = order_products_df \
  .withColumn("qty", col("qty").cast("int"))\
  .withColumn("unit_price", col("price").cast("double"))\
  .withColumnRenamed("id", "product_id")


sales_order_clean = sales_orders_df \
  .withColumn("order_date_key", to_date("order_timestamp"))
              


promotions_clean = promotions_df \
  .withColumn("discount", col("discount").cast("double"))\
  .withColumn("promo_quantity", col("promo_quantity").cast("int"))
  
  

### Join 1 =
order_products + sales_orders 

### WHY INNER JOIN?
------------------------------------------------------------
* We only want valid orders existing in both tables.
* order_products has product rows
* sales_orders has order date/time
* If order exists in products but not in sales_orders:
* bad/incomplete data

#### INNER JOIN keeps only matching rows

### WHY ONLY order_number?
 ------------------------------------------------------------
* order_number is unique order identifier.
* It connects product rows to order header rows.
* customer_id is repetitive and unnecessary here.
* Same customer can place many orders.
* Since order_number identifies exact order,
* joining on it is enough

In [0]:
fact_base = order_products_clean.alias("op") \
  .join(
    sales_order_clean.alias("so"),
    on="order_number",
    how = "inner"
  )

In [0]:
fact_base = fact_base.select(
    col("order_number"),
    col("op.customer_id").alias("customer_id"),
    col("product_id"),
    col("qty"),
    col("unit_price"),
    col("order_date_key")
)

In [0]:
fact_base.display()

### WHY LEFT JOIN?
 ------------------------------------------------------------
* Every sale must remain.
* Promotion may or may not exist

In [0]:
fact_with_promo = fact_base.alias("f") \
  .join(
    promotions_clean.alias("p"),
    on = "order_number",
    how = "left"
  )

### round() 
= is a PySpark function used to round numeric values to a specified number of decimal places.

##### Then round(..., 2) means:
*  keep only 2 digits after decimal

In [0]:
fact_final = fact_with_promo.select(
    col("order_number"),
    col("f.customer_id"),
    col("product_id"),
    col("order_date_key"),
    col("qty"),
    col("unit_price"),
    col("discount"),
    col("promo_quantity"),
    
    round(col("qty") * col("unit_price") * (1 - col("discount")), 2).alias("total_amount")
)

    

In [0]:
fact_final = fact_final.fillna({"discount": 0, "promo_quantity": 0})

In [0]:
fact_final.display()